# ALS baseline - MovieLens 1M


In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "als").is_dir() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT / "als" / "src"))

from datasets import prepare_split
from spark_utils import get_spark_session, stop_spark
from als_utils import run_als
from results_utils import append_result, new_run_id

In [2]:

DATASET       = "ml-1m"
DATA_DIR      = PROJECT_ROOT / "data"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
RESULTS_CSV   = str(PROJECT_ROOT / "results" / "als_results.csv")

K             = 5
LAMBDA        = 0.1
MAX_ITER      = 10
SEED          = 42


CORE_VALUES   = [1, 2, 4]
NUM_RUNS      = 3

# Extra driver heap for local runs on the larger datasets (None = Spark default).
DRIVER_MEMORY = None

Preprocess 

In [3]:
spark = get_spark_session(f"als_{DATASET}_prep", num_cores=CORE_VALUES[0], driver_memory=DRIVER_MEMORY)

TRAIN_PATH, TEST_PATH = prepare_split(spark, DATASET, DATA_DIR, PROCESSED_DIR, seed=SEED)

train = spark.read.parquet(TRAIN_PATH)
test = spark.read.parquet(TEST_PATH)
train_size = train.count()
test_size = test.count()
dataset_size = train_size + test_size
print("train:", train_size, "| test (clean):", test_size,
      "| total:", dataset_size, "| test ratio:", round(test_size / dataset_size, 3))

stop_spark(spark)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/13 14:49:56 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


train: 900038 | test (clean): 100161 | total: 1000199 | test ratio: 0.1


In [4]:
for num_cores in CORE_VALUES:
    print(f"{num_cores} core(s)")
    spark = get_spark_session(f"als_{DATASET}_{num_cores}c", num_cores=num_cores, driver_memory=DRIVER_MEMORY)

    train = spark.read.parquet(TRAIN_PATH).cache()
    test = spark.read.parquet(TEST_PATH).cache()
    train.count(); test.count()

    for run in range(1, NUM_RUNS + 1):
        row = run_als(
            train, test,
            dataset=DATASET, dataset_size=dataset_size, num_cores=num_cores,
            k=K, lam=LAMBDA, max_iter=MAX_ITER, seed=SEED,
            run_id=new_run_id(),
        )
        append_result(row, RESULTS_CSV)
        print(f"  run {run}/{NUM_RUNS}: RMSE={row['test_rmse']:.4f}  train_time={row['train_time']:.2f}s")

    stop_spark(spark)

print("Results appended to", RESULTS_CSV)

1 core(s)


netlib-blas: JNI_OnLoad: dlopen(libblas.so.3) failed: libblas.so.3: cannot open shared object file: No such file or directory
netlib-lapack: JNI_OnLoad: dlopen(liblapack.so.3) failed: liblapack.so.3: cannot open shared object file: No such file or directory


  run 1/3: RMSE=0.8723  train_time=2.14s
  run 2/3: RMSE=0.8723  train_time=1.93s
  run 3/3: RMSE=0.8723  train_time=1.82s
2 core(s)
  run 1/3: RMSE=0.8723  train_time=1.12s
  run 2/3: RMSE=0.8723  train_time=1.15s
  run 3/3: RMSE=0.8723  train_time=1.11s
4 core(s)
  run 1/3: RMSE=0.8723  train_time=0.85s
  run 2/3: RMSE=0.8723  train_time=0.79s
  run 3/3: RMSE=0.8723  train_time=0.76s
Results appended to /mnt/c/Users/89526/Documents/GitHub/ccdpp-pyspark-als-movielens/results/als_results.csv


## Inspect results

In [5]:
import pandas as pd
df = pd.read_csv(RESULTS_CSV)
summary = (df[df.dataset == DATASET]
           .groupby("num_cores")
           .agg(mean_rmse=("test_rmse","mean"),
                mean_train_time=("train_time","mean"),
                runs=("run_id","count"))
           .reset_index())
base = summary.loc[summary.num_cores == summary.num_cores.min(), "mean_train_time"].iloc[0]
summary["speedup"]    = base / summary["mean_train_time"]
summary["efficiency"] = summary["speedup"] / summary["num_cores"]
summary

,num_cores,mean_rmse,mean_train_time,runs,speedup,efficiency
0,1,0.872255,1.962978,3,1.00000,1.000000
1,2,0.872255,1.188409,4,1.65177,0.825885
2,4,0.872255,0.799200,3,2.45618,0.614045
